# 02 · EDGAR collection, and checking an assumption instead of inheriting it

This notebook does two things:

1. shows the EDGAR fetcher and the **observed vs simulated** provenance split;
2. re-derives, from real SEC payloads, the timezone of `acceptanceDateTime`.

The second is the interesting one. The natural reading of that field is wrong,
and being wrong shifts every document by four or five hours — enough to move an
after-hours filing across a session boundary and change which day it could
first be traded on.

Everything here runs **offline** against saved real payloads in
`backend/tests/fixtures/edgar/`. A live cell at the end is optional.

In [ ]:
# Put `backend/` on sys.path so `app.*` imports work regardless of where
# Jupyter was launched from. Walks up until it finds the backend package.
import sys, pathlib

here = pathlib.Path.cwd()
for candidate in (here, *here.parents):
    if (candidate / "backend" / "app").is_dir():
        sys.path.insert(0, str(candidate / "backend"))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("could not locate backend/ — run from inside the repo")

FIXTURES = REPO_ROOT / "backend" / "tests" / "fixtures" / "edgar"
print("repo root :", REPO_ROOT)
print("fixtures  :", FIXTURES, "(exists)" if FIXTURES.is_dir() else "(MISSING)")

## Fetching filings

Network access is *injected*, not imported. That is what lets the whole
collection path be tested against fixtures — and it is also why the live path
needs its own test, as we will see at the end.

In [ ]:
import json
from datetime import datetime
from zoneinfo import ZoneInfo

from app.text_signals.edgar_fetcher import EdgarSubmissionsFetcher, SecUserAgent

ET = ZoneInfo("America/New_York")
AAPL, MSFT = 320193, 789019

def fixture_http(url, headers):
    """Serve the saved payloads instead of calling SEC."""
    assert "User-Agent" in headers, "SEC refuses undeclared clients"
    return (FIXTURES / url.rsplit("/", 1)[-1]).read_bytes()

# The User-Agent is required and has no default: SEC asks for a real contact so
# a misbehaving client can be reached, and a placeholder defeats the purpose.
ua = SecUserAgent(name="Research Notebook", email="research@example.com")

fetcher = EdgarSubmissionsFetcher(
    [AAPL], ua, http_get=fixture_http, form_types=("10-K",)
)
tenks = fetcher(datetime(2015, 1, 1, tzinfo=ET), datetime(2026, 12, 31, tzinfo=ET))

print(f"{len(tenks)} 10-K filings\n")
for f in tenks:
    print(f"{f.accession_number}  {f.acceptance_datetime:%Y-%m-%d %H:%M %Z}  {f.symbol}")

## The question: what timezone is `acceptanceDateTime`?

SEC sends values like `2016-10-26T20:42:16.000Z`. The `Z` says UTC — but EDGAR
is an Eastern-time system whose *filing-date rule* is published in Eastern time,
so "it's really ET, loosely labelled" is a very plausible guess.

There is a way to settle it without documentation, using SEC's own dating rule:

> a submission accepted after **17:30 ET** is *dated* the next business day.

That rule turns `filingDate` into a witness. For each filing we predict the
filing date under both readings and see which one matches reality.

(Section 16 forms — 3/4/5 — are exempt from the cutoff and are excluded.)

In [ ]:
from datetime import date, time, timedelta, timezone

CUTOFF = time(17, 30)
SECTION_16 = {"3", "4", "5"}
UTC = timezone.utc

def rows(cik):
    recent = json.loads((FIXTURES / f"CIK{cik:010d}.json").read_text())["filings"]["recent"]
    return [{k: recent[k][i] for k in recent} for i in range(len(recent["accessionNumber"]))]

def predict_filing_date(moment_et):
    """SEC's rule: after 17:30 ET -> next business day."""
    if moment_et.time() <= CUTOFF:
        return moment_et.date()
    nxt = moment_et.date() + timedelta(days=1)
    while nxt.weekday() >= 5:
        nxt += timedelta(days=1)
    return nxt

def score(cik):
    utc_hit = utc_miss = et_hit = et_miss = 0
    for r in rows(cik):
        if r["form"] in SECTION_16 or not r["acceptanceDateTime"] or not r["filingDate"]:
            continue
        naive = datetime.strptime(r["acceptanceDateTime"], "%Y-%m-%dT%H:%M:%S.%fZ")
        actual = date.fromisoformat(r["filingDate"])
        if predict_filing_date(naive.replace(tzinfo=UTC).astimezone(ET)) == actual:
            utc_hit += 1
        else:
            utc_miss += 1
        if predict_filing_date(naive.replace(tzinfo=ET)) == actual:
            et_hit += 1
        else:
            et_miss += 1
    return utc_hit, utc_miss, et_hit, et_miss

print(f"{'company':8} {'read as UTC':>18} {'read as Eastern':>18}")
for cik, name in ((AAPL, "AAPL"), (MSFT, "MSFT")):
    uh, um, eh, em = score(cik)
    print(f"{name:8} {f'{uh} hit / {um} miss':>18} {f'{eh} hit / {em} miss':>18}")

The UTC reading predicts SEC's own published filing dates; the Eastern reading
does not. That settles it — and note the shape of the evidence: we did not find
a document saying so, we found a *consequence* that only one reading can
produce.

Here is the single cleanest case, where the two readings genuinely disagree.

In [ ]:
from app.text_signals.edgar_fetcher import parse_acceptance_datetime

row = next(r for r in rows(AAPL) if r["accessionNumber"] == "0001628280-16-020309")
print("raw acceptanceDateTime :", row["acceptanceDateTime"])
print("SEC's filingDate       :", row["filingDate"], "(same calendar day)")
print()

as_utc = parse_acceptance_datetime(row["acceptanceDateTime"])
as_eastern = datetime(2016, 10, 26, 20, 42, 16, tzinfo=ET)

print(f"read as UTC     -> {as_utc:%H:%M} ET, cutoff {CUTOFF} -> "
      f"{'before' if as_utc.time() <= CUTOFF else 'after'} -> "
      f"predicts {predict_filing_date(as_utc)}")
print(f"read as Eastern -> {as_eastern:%H:%M} ET, cutoff {CUTOFF} -> "
      f"{'before' if as_eastern.time() <= CUTOFF else 'after'} -> "
      f"predicts {predict_filing_date(as_eastern)}")
print()
print("only the UTC reading reproduces SEC's own filing date")

## Observed vs simulated receipt

Two collection modes that must never be confused:

- `poll()` runs on a schedule from today and **measures** receipt → `OBSERVED`
- `backfill()` reconstructs receipt from publish time plus an assumed polling
  interval → `SIMULATED`

Both are legitimate. Only one is a measurement. A false `OBSERVED` is worse than
an honest `SIMULATED`, because it invites more trust rather than less.

In [ ]:
from app.text_signals.edgar_collector import EdgarCollector, provenance_report

window = (datetime(2015, 1, 1, tzinfo=ET), datetime(2026, 12, 31, tzinfo=ET))
plain = EdgarSubmissionsFetcher([AAPL], ua, http_get=fixture_http)

historical = EdgarCollector(plain).backfill(*window)
now = datetime(2026, 12, 31, 9, 0, tzinfo=ET)
live = EdgarCollector(plain, clock=lambda: now).poll(*window)

print("backfill ->", {r.ingest_time_source.value for r in historical})
print("poll     ->", {r.ingest_time_source.value for r in live})
print()
print("backfill: ingest is publish + assumed polling interval")
r = historical[0]
print(f"  publish {r.publish_time:%Y-%m-%d %H:%M}  ingest {r.ingest_time:%Y-%m-%d %H:%M}")
print()
for key, value in provenance_report(historical + live).items():
    print(f"{key:20} {value}")

`observed_share` is the number that belongs next to any result built on this
corpus. A backfilled study is admissible *because* it is labelled, not in spite
of it.

## Optional: the live path

The cell below calls SEC for real. It is optional and left un-run — but it earns
its place, because the injected-fixture tests structurally cannot exercise real
HTTP, and a bug lived exactly there: the fetcher requests gzip (as SEC asks, to
spare their bandwidth) while `urllib` does not decompress, so `json.loads`
received compressed bytes and failed as what *looked* like an upstream format
change.

Set a real name and email — SEC requires a genuine contact.

In [ ]:
# Uncomment to call SEC for real. Use your own name and email.
#
# live_ua = SecUserAgent(name="Your Name", email="you@example.com")
# live_fetcher = EdgarSubmissionsFetcher([AAPL], live_ua, form_types=("10-K",))
# for f in live_fetcher(datetime(2023, 1, 1, tzinfo=ET), datetime(2024, 12, 31, tzinfo=ET)):
#     print(f.accession_number, f.acceptance_datetime, f.document_url)
print("live cell is commented out by default")